In [ ]:
import os
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

# === FUNCIÓN PARA LIMPIAR TEXTO ===
def limpiar_texto(texto):
    """Elimina emojis o caracteres fuera del BMP (incompatibles con ChromeDriver)."""
    return ''.join(c for c in str(texto) if ord(c) <= 0xFFFF)

# === FUNCIÓN PARA ADJUNTAR ARCHIVO (RESTAURADA A LA VERSIÓN MULTI-ESTRATEGIA) ===
def adjuntar_archivo(driver, ruta_archivo, nombre_empleado, mensaje_adjunto=""):
    """Intenta adjuntar un archivo con múltiples estrategias y un mensaje opcional."""
    
    ruta_archivo = os.path.abspath(ruta_archivo).replace("\\", "/")
    print(f"   📎 Intentando adjuntar: {ruta_archivo}")
    
    try:
        # ESTRATEGIA 1: Buscar el input directamente (la que funcionaba para ti)
        try:
            print("   🔍 Estrategia 1: Buscando input de archivo directamente...")
            file_input = driver.find_element(By.CSS_SELECTOR, "input[type='file'][accept*='image'], input[type='file'][accept*='pdf']")
            file_input.send_keys(ruta_archivo)
            print("   ✅ Archivo cargado con Estrategia 1")
            time.sleep(5)
            # file_input.send_keys(Keys.ENTER)
            enviar_archivo(driver, mensaje_adjunto)
            return True
        except NoSuchElementException:
            print("   ⚠️ Estrategia 1 falló, intentando Estrategia 2...")
        
        # ESTRATEGIA 2: Click en botón de adjuntar + input
        try:
            print("   🔍 Estrategia 2: Buscando botón de clip...")
            selectores_clip = [
                "//div[@title='Adjuntar']",
                "//span[@data-icon='plus' or @data-icon='clip' or @data-icon='attach-menu-plus']",
                "//button[contains(@aria-label, 'Adjuntar')]",
                "//div[@role='button' and .//span[@data-icon='attach-menu-plus']]"
            ]
            clip_button = None
            for selector in selectores_clip:
                try:
                    clip_button = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, selector)))
                    break
                except: continue
            
            if not clip_button: raise Exception("No se encontró el botón de adjuntar")
            
            clip_button.click()
            time.sleep(2)
            
            file_input = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "input[type='file']")))
            file_input.send_keys(ruta_archivo)
            print("   ✅ Archivo cargado con Estrategia 2")
            time.sleep(5)
            enviar_archivo(driver, mensaje_adjunto)
            return True
        except Exception as e:
            print(f"   ⚠️ Estrategia 2 falló: {e}")
        
        return False
        
    except Exception as e:
        print(f"   ❌ Error general al adjuntar archivo: {e}")
        return False


# === FUNCIÓN PARA ENVIAR ARCHIVO (VERSIÓN SIMPLIFICADA) ===
# === FUNCIÓN PARA ENVIAR ARCHIVO (VERSIÓN FINAL: CLIC AL BOTÓN) ===
def enviar_archivo(driver, mensaje_adjunto=""):
    """
    Encuentra y hace clic en el botón 'Enviar' final en la pantalla de vista previa del adjunto.
    Este es el método más fiable y similar a la acción humana.
    """
    # Aumentamos la espera para dar tiempo a que la carga del archivo se complete
    # y la UI de WhatsApp se actualice completamente.
    wait_time = 20 
    print(f"   📤 Esperando hasta {wait_time} segundos a que aparezca el botón de envío final...")

    try:
        # Este selector busca el botón que contiene el ícono de 'enviar'.
        # Es el selector más robusto para el botón verde de envío de adjuntos.
        send_button_xpath = "//button[.//span[@data-icon='send']]"
        
        send_button = WebDriverWait(driver, wait_time).until(
            EC.element_to_be_clickable((By.XPATH, send_button_xpath))
        )
        
        print("   ✅ Botón de envío final encontrado.")
        
        # Usamos un clic de JavaScript para la máxima fiabilidad, evitando intercepciones.
        driver.execute_script("arguments[0].click();", send_button)
        
        print("   ✅ Archivo enviado con clic en el botón de envío.")
        time.sleep(4) # Damos tiempo para que el envío se procese.
        return True

    except Exception as e:
        print(f"   ❌ No se pudo encontrar o hacer clic en el botón de envío final. Error: {e}")
        return False

# === CONFIGURACIÓN ===
ruta_excel = r"C:\Users\Dataa\Desktop\empleados.xlsx"
codigo_pais = "57"
columna_estado = "estado"

# === CARGAR DATOS ===
try:
    df = pd.read_excel(ruta_excel)
    pendientes = df[df[columna_estado].str.lower() == "pendiente"]
    print(f"Se encontraron {len(pendientes)} empleados pendientes.")
except Exception as e:
    print(f"❌ No se pudo leer el archivo Excel: {e}")
    raise

# === INICIAR WHATSAPP WEB ===
driver = webdriver.Chrome()
driver.get("https://web.whatsapp.com/")



# === ENVIAR MENSAJES Y PDF (LÓGICA ORIGINAL RESTAURADA) ===
for i, row in pendientes.iterrows():
    celular = str(row["celular"]).strip().replace("+", "")
    link_pdf = str(row["link"]).strip()
    empleado = str(row["empleado"]).capitalize()

    if not os.path.exists(link_pdf):
        print(f"⚠️ No se encontró el archivo: {link_pdf}")
        df.at[i, columna_estado] = "Error: Archivo no existe"
        continue

    # Recrear el mensaje original
    nombre_pdf = os.path.basename(link_pdf)
    mensaje_original = (
        f"Hola {empleado}, te envío tu archivo en PDF.\n"
        # f"Por favor confirma la recepción.\n\n"
        # f"Archivo adjunto: {nombre_pdf}"
    )
    mensaje = limpiar_texto(mensaje_original)

    print(f"\n📤 Enviando a {empleado} ({celular})...")

    # Abrir chat
    driver.get(f"https://web.whatsapp.com/send?phone={codigo_pais}{celular}")
    
    try:
        # 1. ENVIAR MENSAJE DE TEXTO PRIMERO
        chat_box = WebDriverWait(driver, 20).until(
            EC.presence_of_element_located(
                (By.XPATH, "//div[@contenteditable='true' and @data-tab='10']")
            )
        )
        chat_box.send_keys(mensaje)
        chat_box.send_keys(Keys.ENTER)
        print("   ✅ Mensaje de texto enviado.")
        time.sleep(3) # Espera para que el mensaje se envíe

        # 2. ADJUNTAR Y ENVIAR EL ARCHIVO
        if adjuntar_archivo(driver, link_pdf, empleado, mensaje_adjunto=""):
            print(f"✅ Archivo enviado a {empleado}")
            df.at[i, columna_estado] = "Enviado"
        else:
            print(f"⚠️ Falló el envío del archivo a {empleado}")
            df.at[i, columna_estado] = "Enviado (solo texto)"

    except Exception as e:
        print(f"❌ Error general al procesar a {empleado}: {e}")
        df.at[i, columna_estado] = f"Error: {str(e)[:50]}"

    time.sleep(5)

# === GUARDAR RESULTADOS ===
try:
    df.to_excel(ruta_excel, index=False)
    print("\n✅ Proceso completado. Los estados fueron actualizados en el Excel.")
except Exception as e:
    print(f"❌ No se pudo guardar el archivo Excel: {e}")







In [ ]:
# driver.quit()